In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

C:\Users\Abhinesh Singh\AppData\Local\Temp\ipykernel_16608\3799568052.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\RAG\RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 1: Load the dataset and split the data

loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)


embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorStore = FAISS.from_documents(chunks,embedding_model)
retriever = vectorStore.as_retriever(search_type="mmr",search_kwargs={"k":4, "lambda_mult":0.7})


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4318.45it/s]


In [ ]:
#step 2 llm initization
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)
llm

ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.8'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000187CD14CF50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000187CD17B050>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
# Step 3: Query Decomposition 
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:                                                    
""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [5]:
query = "How does Langchain use memory and agents compared to CrewAI?"
decomposition_question = decomposition_chain.invoke({"question":query})

In [6]:
print(decomposition_question)

To better understand and retrieve relevant documents, the complex question can be decomposed into the following sub-questions:

1. **What is Langchain's approach to memory management?** 
   - This sub-question focuses on how Langchain utilizes memory, which could include details on data storage, retrieval, and any specific memory-related features.

2. **How does Langchain implement and utilize agents?** 
   - This sub-question delves into the role and functionality of agents within Langchain, including how they interact with memory and other components of the system.

3. **What are CrewAI's methods for memory management and agent implementation?** 
   - This sub-question shifts the focus to CrewAI, seeking information on its memory management strategies and how it uses agents, providing a basis for comparison with Langchain.

4. **How do Langchain and CrewAI differ in their use of memory and agents?** 
   - This sub-question directly addresses the comparative aspect of the original que

In [8]:
# QA chain per sub-question

qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question : {input}
""")

qa_chain = create_stuff_documents_chain(llm = llm , prompt=qa_prompt)

In [9]:
# Step-5: Full RAG pipeline logic

def full_query_decomposition_rag_pipeline(user_query):

    # Decompose the query
    sub_qs_text = decomposition_chain.invoke(
        {"question": user_query}
    )

    sub_questions = [
        q.strip(" -01234567890.").strip()
        for q in sub_qs_text.split("\n")
        if q.strip()
    ]

    results = []

    for subq in sub_questions:
        docs = retriever.invoke(subq)

        result = qa_chain.invoke({
            "input": subq,
            "context": docs
        })

        results.append(f"Q: {subq}\nA: {result}")

    return "\n\n".join(results)

In [14]:
# Step 6: Run

query = "How does LangChain implement RAG using vector databases, and how can CrewAI integrate with LangChain in a multi-agent system?"

final_answer = full_query_decomposition_rag_pipeline(query)

print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: To decompose the complex question into smaller sub-questions for better document retrieval, let's break it down as follows:
A: It appears that you want to break down a complex question into smaller sub-questions for better document retrieval, likely using LangChain. To do this effectively, let's consider the following steps and sub-questions based on the capabilities of LangChain:

1. **Identify the Goal**: What is the primary objective of the document retrieval? Is it for Retrieval-Augmented Generation (RAG), semantic search, or another purpose?

2. **Choose Retrieval Methods**: Should we use keyword-based (sparse) retrieval, embedding-based (dense) retrieval, or a hybrid approach? Given LangChain's support for hybrid retrieval, combining both methods could be beneficial for catching exact term matches and semantically similar content.

3. **Select Vector Databases**: If embedding-based retrieval is chosen, which vector database should be used (e.g., FAISS, Chroma,

In [13]:
# Question 2
query = "How do LangChain and CrewAI differ in agent orchestration, tool usage, and workflow management?"

final_answer = full_query_decomposition_rag_pipeline(query)

print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: To better understand and retrieve relevant documents, the complex question can be decomposed into the following smaller sub-questions:
A: To better understand and retrieve relevant documents, the complex question can be decomposed into the following smaller sub-questions:

1. What is the main topic of the document?
2. What are the key concepts related to the topic?
3. What are the specific keywords or phrases associated with the topic?
4. How can semantic search be applied to retrieve relevant documents?
5. What role do vector databases play in facilitating semantic search?

By breaking down the complex question into these smaller sub-questions, it becomes easier to understand the context and identify the relevant information, such as the importance of LangChain's integration with vector databases like FAISS, Chroma, Pinecone, and Weaviate, and the usefulness of CrewAI in multi-step workflows.

Q: **What are the key differences in agent orchestration between LangCha

```mermaid
flowchart TD

    A([Start])

    B[Receive User Query]

    C[decomposition_chain.invoke]

    D[Generate Sub-Questions]

    E{{For Each Sub-Question}}

    F[retriever.invoke]

    G[Retrieve Relevant Documents]

    H[qa_chain.invoke]

    I[Generate Answer]

    J[Append Result to List]

    K{{More Sub-Questions?}}

    L[Join All Results]

    M[Return Final Answer]

    N([End])

    A --> B
    B --> C
    C --> D
    D --> E

    E --> F
    F --> G
    G --> H
    H --> I
    I --> J
    J --> K

    K -- Yes --> E
    K -- No --> L

    L --> M
    M --> N
```